# Experiment: Run DA3 Metric Large on a free Colab GPU

**Goal:** Measure DA3's speed and depth quality on the same nine images used by the laptop pilot, then estimate the time needed for all **5,686** coursework frames.

1. Open this notebook in [Colab](https://colab.research.google.com/).
2. Choose **Runtime > Change runtime type > GPU**, and **Runtime version 2026.07** (Python 3.12).
3. Run the cells in order. Upload **depth_model_comparison_inputs.zip** when prompted.
4. Download the results ZIP before ending the session.

An available Colab GPU and Google account are needed. No UCL CS account is required. The first cell stops before downloads if CUDA is unavailable. The notebook downloads only DA3 Metric Large: **1.337 GB of weights plus its 847-byte configuration**, along with small dependencies and its pinned official source. It does not purchase compute, mount Drive or fall back to CPU.

Two sequential cases measure **FP32** and **FP16** on three spread frames each from Replica, TUM and KITTI. Six images have depth GT; KITTI has none. This is a small diagnostic pilot, not full-sequence depth evaluation or a SLAM result. GPU timings have not been measured here; the cells below produce them in your session.

References: [Colab runtime versions](https://research.google.com/colaboratory/runtime-version-faq.html), [Colab limits](https://research.google.com/colaboratory/faq.html), [official DA3](https://github.com/ByteDance-Seed/Depth-Anything-3), [metric model](https://huggingface.co/depth-anything/DA3METRIC-LARGE).

## 1. Check Python and the GPU

Use runtime 2026.07. The probe runs in a separate process and records the assigned GPU. FP32 disables NVIDIA TF32; FP16 is measured as a separate configuration.

In [ ]:
import datetime
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys

if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Select Runtime > Change runtime type > Runtime version 2026.07, then reconnect.')
probe_code = """import json, torch
available = torch.cuda.is_available()
print(json.dumps({'cuda_available': available, 'torch': torch.__version__, 'cuda_version': torch.version.cuda,
                  'gpu': torch.cuda.get_device_name(0) if available else None,
                  'gpu_memory_gib': torch.cuda.get_device_properties(0).total_memory / 2**30 if available else None}))
"""
gpu_info = json.loads(subprocess.check_output([sys.executable, '-c', probe_code], text=True))
if not gpu_info['cuda_available']:
    raise RuntimeError('No CUDA GPU allocation. Stop here and try a GPU runtime later; do not run the cases on CPU.')
if shutil.disk_usage('/content').free < 6 * 1024**3:
    raise RuntimeError('At least 6 GiB of free temporary disk is needed for weights, environments and outputs.')
run_env = {**os.environ, 'PYTHONDONTWRITEBYTECODE': '1', 'NVIDIA_TF32_OVERRIDE': '0',
           'MPLBACKEND': 'Agg', 'TOKENIZERS_PARALLELISM': 'false'}
nvidia_info = subprocess.check_output(['nvidia-smi'], text=True)
print(json.dumps(gpu_info, indent=2))
print(nvidia_info)

## 2. Upload the shared nine-frame input bundle

Select **depth_model_comparison_inputs.zip**. Its manifest retains original indices, timestamps, calibration and GT associations. No full dataset or model checkpoint needs uploading.

In [ ]:
from collections import Counter
from google.colab import files
import io
import stat
import zipfile

uploaded = files.upload()
archives = [name for name in uploaded if name.endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Upload exactly one ZIP: depth_model_comparison_inputs.zip')
workspace = Path('/content/cw1_da3_smoke_run')
workspace.mkdir(parents=True, exist_ok=True)
input_archive_sha256 = hashlib.sha256(uploaded[archives[0]]).hexdigest()
with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as archive:
    entries = archive.infolist()
    if sum(info.file_size for info in entries) > 50 * 1024**2:
        raise ValueError('The selected-input bundle should expand to less than 50 MiB')
    if len({info.filename for info in entries}) != len(entries):
        raise ValueError('Duplicate archive paths are not accepted')
    for info in entries:
        member = Path(info.filename)
        destination = (workspace / member).resolve()
        if (member.is_absolute() or '..' in member.parts or not member.parts
                or member.parts[0] != 'cw1_depth_model_comparison'
                or not destination.is_relative_to(workspace.resolve())
                or stat.S_ISLNK(info.external_attr >> 16)):
            raise ValueError(f'Unexpected archive member: {info.filename}')
    archive.extractall(workspace)
project = workspace / 'cw1_depth_model_comparison'
manifest_path = project / 'sample_manifest.json'
expected_manifest_sha256 = '1345e08be0c92f6a125b7b9271e28a8c0d93ea7f2d9481849ddcc349b51ec8f0'
if hashlib.sha256(manifest_path.read_bytes()).hexdigest() != expected_manifest_sha256:
    raise ValueError('The uploaded selection differs from the shared CPU/GPU comparison manifest')
sample_manifest = json.loads(manifest_path.read_text())
expected_indices = {'replica_room0': [200, 1000, 1799], 'tum_fr3': [258, 1292, 2326], 'kitti06': [110, 550, 990]}
if Counter(row['sequence'] for row in sample_manifest) != Counter({name: 3 for name in expected_indices}):
    raise ValueError('Expected exactly three frames from each sequence')
for name, indices in expected_indices.items():
    if [row['index'] for row in sample_manifest if row['sequence'] == name] != indices:
        raise ValueError(f'Original frame indices differ for {name}')
for name in ('benchmark_models.py', 'model_adapters.py', 'da3_adapter.py', 'summarize_model_benchmarks.py'):
    if not (project / 'scripts/part2' / name).is_file():
        raise FileNotFoundError(f'Missing {name}; regenerate the current comparison bundle')
results_root = project / 'results/da3_gpu_smoke'
results_root.mkdir(parents=True, exist_ok=True)
(results_root / 'sample_manifest.json').write_bytes(manifest_path.read_bytes())
(results_root / 'nvidia_smi.txt').write_text(nvidia_info)
(results_root / 'runtime.json').write_text(json.dumps({
    'started_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'base_gpu': gpu_info,
    'input_archive_sha256': input_archive_sha256, 'sample_manifest_sha256': expected_manifest_sha256,
    'tf32_override': '0', 'selection': expected_indices,
}, indent=2))
print('Input bundle verified:', len(sample_manifest), 'frames; SHA256', input_archive_sha256)

## 3. Create an isolated environment that inherits Colab CUDA

The notebook preserves Colab's PyTorch/Torchvision and installs only the support packages needed by the adapter and reports. It uses the metric network and official image processor directly; DA3's full app dependencies, xformers and Open3D are not installed.

In [ ]:
env_root = Path('/content/cw1_da3_smoke_env')
python = env_root / 'bin/python'
if not python.is_file():
    subprocess.run([sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages', str(env_root)], check=True)
subprocess.run([str(python), '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check',
                'numpy==1.26.4', 'opencv-python-headless==4.10.0.84', 'timm==1.0.30',
                'scipy', 'pandas', 'pyyaml', 'matplotlib', 'einops',
                'omegaconf', 'addict', 'imageio', 'tqdm', 'safetensors', 'huggingface-hub'], check=True)
env_gpu_info = json.loads(subprocess.check_output([str(python), '-c', probe_code], text=True, env=run_env))
if (not env_gpu_info['cuda_available'] or env_gpu_info['torch'] != gpu_info['torch']
        or env_gpu_info['cuda_version'] != gpu_info['cuda_version']):
    raise RuntimeError('CUDA/PyTorch changed unexpectedly; stop and inspect this environment')
subprocess.run([str(python), '-c',
                'import numpy, cv2, torch, torchvision; assert numpy.__version__ == "1.26.4"; '
                'print("NumPy", numpy.__version__, "OpenCV", cv2.__version__, "CUDA", torch.cuda.is_available())'],
               check=True, env=run_env)
freeze = subprocess.check_output([str(python), '-m', 'pip', 'freeze'], text=True)
(results_root / 'environment.txt').write_text(freeze)
(results_root / 'venv_gpu.json').write_text(json.dumps(env_gpu_info, indent=2))

## 4. Fetch the pinned official DA3 source

The source revision is fixed. No editable package installation or automatic model download is used by this adapter.

In [ ]:
source_root = Path('/content/cw1_da3_smoke_source')
source_url = 'https://github.com/ByteDance-Seed/Depth-Anything-3.git'
source_commit = '3d835ec1a5802d64a8b8b15f817a1ab54809bfe4'
if not (source_root / '.git').is_dir():
    if source_root.exists() and any(source_root.iterdir()):
        raise RuntimeError(f'Unexpected nonempty source folder: {source_root}')
    subprocess.run(['git', 'init', str(source_root)], check=True)
    subprocess.run(['git', '-C', str(source_root), 'remote', 'add', 'origin', source_url], check=True)
present = subprocess.run(['git', '-C', str(source_root), 'cat-file', '-e', source_commit + '^{commit}'],
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
if not present:
    subprocess.run(['git', '-C', str(source_root), 'fetch', '--depth', '1', 'origin', source_commit], check=True)
subprocess.run(['git', '-C', str(source_root), 'checkout', '--detach', source_commit], check=True)
actual = subprocess.check_output(['git', '-C', str(source_root), 'rev-parse', 'HEAD'], text=True).strip()
if actual != source_commit:
    raise RuntimeError('DA3 source revision differs from the pinned version')
(results_root / 'source_commit.json').write_text(json.dumps({'url': source_url, 'commit': source_commit}, indent=2))
print('Pinned DA3 source:', actual)

## 5. Download and verify DA3 Metric Large

The official Hugging Face URLs use an immutable revision. Both files are checked against their expected byte count and SHA256. Interrupted `.partial` downloads resume when this cell is rerun; checksum failures stop before inference.

In [ ]:
weights_root = Path('/content/cw1_da3_smoke_weights')
weights_root.mkdir(exist_ok=True)
da3_revision = '4010e39f3634a45bc60553321fb49fb760bd594e'
download_specs = [
    ('da3_config', f'https://huggingface.co/depth-anything/DA3METRIC-LARGE/resolve/{da3_revision}/config.json',
     'da3metric-large/config.json', 847, 'a336f3e76fe375aaae17a9aed9130c9f2aa061535d317ec57dcb2f1f02e1dd53'),
    ('da3_weights', f'https://huggingface.co/depth-anything/DA3METRIC-LARGE/resolve/{da3_revision}/model.safetensors',
     'da3metric-large/model.safetensors', 1336734448, 'bbea5b0b3ee389849cffa7ddae89de064a90abd2b055fc5aa99aac68db324776'),
]

def digest_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024**2), b''):
            digest.update(block)
    return digest.hexdigest()

def download_checked(url, relative, expected_bytes, expected_sha):
    target = weights_root / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_file():
        candidate = target
    else:
        candidate = target.with_name(target.name + '.partial')
        if candidate.exists() and candidate.stat().st_size > expected_bytes:
            raise ValueError(f'Oversized partial download: {candidate}; inspect it before retrying')
        if not candidate.is_file() or candidate.stat().st_size != expected_bytes:
            subprocess.run(['curl', '-fL', '--retry', '3', '--continue-at', '-', url, '-o', str(candidate)], check=True)
    if candidate.stat().st_size != expected_bytes:
        raise ValueError(f'Incomplete {relative}; rerun this cell to resume')
    checksum = digest_file(candidate)
    if checksum != expected_sha:
        raise ValueError(f'SHA256 mismatch for {relative}; do not use this file')
    if candidate != target:
        candidate.replace(target)
    return target, checksum

checkpoint_paths, checkpoint_records = {}, []
for key, url, relative, expected_bytes, expected_sha in download_specs:
    target, checksum = download_checked(url, relative, expected_bytes, expected_sha)
    checkpoint_paths[key] = target
    checkpoint_records.append({'key': key, 'url': url, 'file': relative, 'bytes': expected_bytes, 'sha256': checksum})
    (results_root / 'checkpoints.json').write_text(json.dumps(checkpoint_records, indent=2))
    print('Verified', relative, f'({expected_bytes / 1024**2:.1f} MiB)', flush=True)
da3_config = json.loads(checkpoint_paths['da3_config'].read_text())
if da3_config.get('model_name', '').lower() != 'da3metric-large':
    raise ValueError('The DA3 checkpoint is not the required metric model')

## 6. Check the exact portable inputs

This inventory checks all nine RGB dimensions against calibration and the six available GT matches without loading a model. The indices must match the laptop pilot.

In [ ]:
inventory_code = r"""
import json, sys
from pathlib import Path
import cv2
sys.path.insert(0, 'scripts')
from part2.benchmark_models import load_samples
samples = load_samples(('replica_room0','tum_fr3','kitti06'), sample_manifest=Path(sys.argv[1]))
rows=[]
for sample in samples:
    frame,camera=sample['frame'],sample['camera']
    image=cv2.imread(str(frame.rgb))
    assert image is not None and image.shape[:2] == (camera.height,camera.width)
    if frame.depth_gt is not None:
        gt=cv2.imread(str(frame.depth_gt),cv2.IMREAD_UNCHANGED)
        assert gt is not None and gt.shape == image.shape[:2]
        assert abs(frame.depth_timestamp-frame.timestamp) <= .02
    rows.append({'sequence':sample['sequence'],'index':frame.index,'timestamp':frame.timestamp,
                 'rgb':frame.rgb.name,'gt_available':frame.depth_gt is not None,
                 'size':[camera.width,camera.height]})
assert len(rows)==9 and sum(row['gt_available'] for row in rows)==6
print(json.dumps(rows))
"""
inventory = json.loads(subprocess.check_output([str(python), '-c', inventory_code, str(manifest_path)],
                                               cwd=project, env=run_env, text=True))
(results_root / 'input_inventory.json').write_text(json.dumps(inventory, indent=2))
print(json.dumps(inventory, indent=2))

## 7. Run FP32 and FP16, one model process at a time

Each case saves a prediction and JSON record after each frame. A separate warmup is recorded before timing. Inference time includes preprocessing, synchronized network execution and postprocessing; it excludes image reads, saving, GT scoring and model startup.

The cases use identical inputs and fixed processing resolution/resize policy. If one precision fails or exceeds GPU memory, its error remains in the logs and the other case still runs; settings and device are never changed silently. Rerunning in the **same surviving Colab VM** reuses matching, checksummed completed predictions. A runtime reset can erase temporary storage, so download partial work before ending the session.

In [ ]:
benchmark_root = results_root / 'benchmarks'
logs_root = results_root / 'logs'
benchmark_root.mkdir(exist_ok=True)
logs_root.mkdir(exist_ok=True)
cases = []
for precision in ('float32','float16'):
    label = f'spread_da3_metric_large_cuda_{precision}'
    cases.append({'label': label, 'model': 'da3_metric_large', 'precision': precision,
                  'source_root': str(source_root), 'checkpoint': str(checkpoint_paths['da3_weights'].parent),
                  'sequences': ['replica_room0','tum_fr3','kitti06'], 'process_res': 504, 'domain': None,
                  'output': str(benchmark_root / (label + '.json')),
                  'pred_root': str(project / 'depth_maps/da3_gpu_smoke' / label)})
(results_root / 'case_plan.json').write_text(json.dumps(cases, indent=2))
print('Two CUDA cases, nine timed frames each, plus one warmup per case.')

In [ ]:
import time

ONLY_CASES = []  # Example: ['spread_da3_metric_large_cuda_float16']; empty means both cases.
unknown = set(ONLY_CASES) - {case['label'] for case in cases}
if unknown:
    raise ValueError(f'Unknown cases: {sorted(unknown)}')
selected_cases = [case for case in cases if not ONLY_CASES or case['label'] in ONLY_CASES]
progress_path = results_root / 'case_progress.json'
progress = json.loads(progress_path.read_text()) if progress_path.exists() else {}
for case in selected_cases:
    command = [str(python), str(project / 'scripts/part2/benchmark_models.py'),
               '--model', case['model'], '--device', 'cuda', '--precision', case['precision'],
               '--source-root', case['source_root'], '--checkpoint', case['checkpoint'],
               '--sample-manifest', str(manifest_path), '--output', case['output'],
               '--pred-root', case['pred_root']]
    for sequence in case['sequences']:
        command += ['--sequence', sequence]
    if case['domain'] is not None:
        command += ['--domain', case['domain']]
    if case['process_res'] is not None:
        command += ['--process-res', str(case['process_res'])]
    log_path = logs_root / (case['label'] + '.log')
    print('Starting', case['label'], flush=True)
    tick = time.perf_counter()
    with log_path.open('a') as log:
        log.write('\nCASE ATTEMPT ' + datetime.datetime.now(datetime.timezone.utc).isoformat() + '\n')
        log.flush()
        process = subprocess.run(command, cwd=project, env=run_env, stdout=log, stderr=subprocess.STDOUT)
    elapsed = time.perf_counter() - tick
    record = {'returncode': process.returncode, 'whole_process_seconds': elapsed,
              'command': command, 'log': str(log_path), 'finished_utc': datetime.datetime.now(datetime.timezone.utc).isoformat()}
    output_path = Path(case['output'])
    if output_path.is_file():
        benchmark = json.loads(output_path.read_text())
        record['benchmark_status'] = benchmark.get('status')
        record['frames_ok'] = sum(row.get('status') == 'ok' for row in benchmark.get('frames', []))
        record['frames_recorded'] = len(benchmark.get('frames', []))
    progress.setdefault(case['label'], []).append(record)
    temporary = progress_path.with_suffix('.tmp')
    temporary.write_text(json.dumps(progress, indent=2))
    temporary.replace(progress_path)
    print('Finished', case['label'], 'returncode', process.returncode,
          f'wall={elapsed:.1f}s', 'frames_ok=', record.get('frames_ok', 0), flush=True)
    if process.returncode != 0:
        print('Failure retained; next case will still run. Last log lines:')
        print('\n'.join(log_path.read_text(errors='replace').splitlines()[-15:]))
print('Case loop finished. Continue to summary/export even when some cases failed.')

## 8. Compare quality and estimate the full workload

The table reports raw metric-depth error, median-scaled shape error and coverage on Replica/TUM. KITTI has no supplied depth GT. FP32 and FP16 remain separate.

Full-workload estimates weight each sequence's measured seconds per frame by its actual count: **Replica 2,000 + TUM 2,585 + KITTI 1,101 = 5,686 frames**. These are planning estimates, excluding image I/O, saving and startup, rather than measured full-sequence runtimes. A partial/failed case receives no full-workload estimate.

In [ ]:
import csv
import statistics

summary_root = results_root / 'summary'
subprocess.run([str(python), str(project / 'scripts/part2/summarize_model_benchmarks.py'),
                '--input-root', str(benchmark_root), '--output-root', str(summary_root), '--no-plot'],
               cwd=project, env=run_env, check=True)
frame_counts = {'replica_room0': 2000, 'tum_fr3': 2585, 'kitti06': 1101}
assert sum(frame_counts.values()) == 5686
estimates = []
for case in cases:
    path = Path(case['output'])
    document = json.loads(path.read_text()) if path.is_file() else {}
    case_rows = []
    for sequence, count in frame_counts.items():
        rows = [row for row in document.get('frames',[]) if row.get('sequence') == sequence]
        complete = len(rows) == 3 and all(row.get('status') == 'ok' for row in rows)
        seconds = statistics.mean(row['inference_seconds'] for row in rows) if complete else None
        case_rows.append({'case': case['label'], 'sequence': sequence, 'gpu': env_gpu_info['gpu'],
                          'precision': case['precision'], 'frames_measured': sum(row.get('status') == 'ok' for row in rows),
                          'frames_total': count, 'seconds_per_frame': seconds,
                          'estimated_hours': seconds * count / 3600 if seconds is not None else None})
    estimates.extend(case_rows)
    full_hours = (sum(row['estimated_hours'] for row in case_rows)
                  if all(row['estimated_hours'] is not None for row in case_rows) else None)
    estimates.append({'case': case['label'], 'sequence': 'all_5686', 'gpu': env_gpu_info['gpu'],
                      'precision': case['precision'], 'frames_measured': sum(row['frames_measured'] for row in case_rows),
                      'frames_total': 5686, 'seconds_per_frame': None, 'estimated_hours': full_hours})
(results_root / 'runtime_estimates.json').write_text(json.dumps(estimates, indent=2))
with (results_root / 'runtime_estimates.csv').open('w',newline='') as handle:
    writer = csv.DictWriter(handle,fieldnames=list(estimates[0]))
    writer.writeheader();writer.writerows(estimates)
from IPython.display import Markdown, display
display(Markdown((summary_root / 'comparison.md').read_text()))
for row in estimates:
    hours = f"{row['estimated_hours']:.3f} h" if row['estimated_hours'] is not None else 'unavailable (partial/failed)'
    seconds = f"{row['seconds_per_frame']:.4f} s/frame; " if row['seconds_per_frame'] is not None else ''
    print(row['precision'], row['sequence'], seconds + hours)

## 9. View depth predictions on a common metre scale

The first selected frame from each sequence is displayed. Each contact sheet uses the same colour range for GT and both precision cases; large values are clipped only for display. Visual plausibility on KITTI does not establish numerical accuracy.

In [ ]:
plot_code = r"""
import json, math, sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

project, results = map(Path, sys.argv[1:])
cases = json.loads((results/'case_plan.json').read_text())
manifest = json.loads((project/'sample_manifest.json').read_text())
figures = results/'figures'
figures.mkdir(exist_ok=True)
loaded = {}
for case in cases:
    path = Path(case['output'])
    if path.is_file():
        loaded[case['label']] = json.loads(path.read_text())

def label(case):
    name = {'da3_metric_large':'DA3 Metric Large'}[case['model']]
    return name + (' FP16' if case['precision']=='float16' else ' FP32')

fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for ax, sequence in zip(axes, ('replica_room0','tum_fr3','kitti06')):
    points=[]
    for case in cases:
        document = loaded.get(case['label'])
        if document is None or sequence not in case['sequences']:
            continue
        runtime = document.get('signature',{}).get('runtime',{})
        if not str(runtime.get('device','')).startswith('cuda'):
            continue
        rows = [row for row in document.get('frames',[]) if row.get('sequence')==sequence]
        if len(rows)!=3 or any(row.get('status')!='ok' for row in rows):
            continue
        seconds = float(np.mean([row['inference_seconds'] for row in rows]))
        if sequence=='kitti06':
            points.append((label(case),seconds,None))
        elif all(row.get('raw') and row['raw'].get('prediction_coverage')==1 for row in rows):
            points.append((label(case),seconds,float(np.mean([row['raw']['abs_rel'] for row in rows]))))
    if not points:
        ax.text(.5,.5,'No complete GPU cases',ha='center',va='center',transform=ax.transAxes)
    elif sequence=='kitti06':
        ax.barh([p[0] for p in points],[p[1] for p in points])
        ax.set_xlabel('Mean inference seconds/frame')
        ax.invert_yaxis()
    else:
        for title, seconds, error in points:
            ax.scatter(seconds,error)
            ax.annotate(title,(seconds,error),xytext=(4,4),textcoords='offset points',fontsize=8)
        ax.set_xscale('log')
        ax.set_xlabel('Mean inference seconds/frame (log scale)')
        ax.set_ylabel('Raw AbsRel (lower is better)')
        ax.grid(alpha=.25)
    ax.set_title(sequence + (' - no depth GT' if sequence=='kitti06' else ' - 3 frames, full GT-pixel coverage'))
fig.suptitle('Measured GPU configurations only; FP32 and FP16 remain separate')
fig.tight_layout()
fig.savefig(figures/'gpu_quality_time.png',dpi=150)
plt.close(fig)

for sequence in ('replica_room0','tum_fr3','kitti06'):
    item=next(row for row in manifest if row['sequence']==sequence)
    rgb=cv2.imread(str(project/item['rgb']))
    panels=[('RGB',cv2.cvtColor(rgb,cv2.COLOR_BGR2RGB),'rgb')]
    gt=None
    if item['depth_gt'] is not None:
        raw=cv2.imread(str(project/item['depth_gt']),cv2.IMREAD_UNCHANGED)
        gt=raw.astype(float)/item['gt_depth_factor']
        gt[raw==0]=np.nan
        panels.append(('Ground truth (m)',gt,'depth'))
    for case in cases:
        if sequence not in case['sequences']:
            continue
        document=loaded.get(case['label'],{})
        row=next((r for r in document.get('frames',[]) if r.get('sequence')==sequence and r.get('index')==item['index']),None)
        prediction=None
        if row and row.get('status')=='ok' and Path(row['prediction']).is_file():
            prediction=np.load(row['prediction'],allow_pickle=False)
        panels.append((label(case),prediction,'depth'))
    reference=gt
    if reference is None:
        arrays=[array[np.isfinite(array)&(array>0)] for _,array,kind in panels if kind=='depth' and array is not None]
        reference=np.concatenate(arrays) if arrays else np.array([1.])
    finite=reference[np.isfinite(reference)&(reference>0)]
    vmax=float(np.percentile(finite,98)) if finite.size else 1.
    vmax=max(vmax,.001)
    norm=Normalize(vmin=0,vmax=vmax,clip=True)
    rows=math.ceil(len(panels)/3)
    fig,axes=plt.subplots(rows,3,figsize=(15,rows*3.5),squeeze=False,layout='constrained')
    image_handle=None
    for ax,(title,array,kind) in zip(axes.flat,panels):
        if array is None:
            ax.text(.5,.5,'Unavailable / failed',ha='center',va='center',transform=ax.transAxes)
        elif kind=='rgb':
            ax.imshow(array)
        else:
            image_handle=ax.imshow(array,cmap='viridis',norm=norm)
        ax.set_title(title);ax.axis('off')
    for ax in list(axes.flat)[len(panels):]: ax.axis('off')
    if image_handle is not None:
        fig.colorbar(image_handle,ax=list(axes.flat),label='Camera-z depth (m)',shrink=.7)
    fig.suptitle(f'{sequence}, original frame {item["index"]}; shared range 0 to {vmax:.2f} m (display clipped)')
    fig.savefig(figures/f'depth_predictions_{sequence}.png',dpi=120)
    plt.close(fig)
print(json.dumps([str(path) for path in sorted(figures.glob('*.png'))]))
"""
figure_paths = json.loads(subprocess.check_output([str(python), '-c', plot_code, str(project), str(results_root)],
                                                 cwd=project, env=run_env, text=True))
from IPython.display import Image
for figure in figure_paths:
    display(Image(filename=figure))

## 10. Download the measured results

Run this cell before ending the runtime, including after interrupting the case loop. The ZIP contains predictions, per-frame results, errors/logs, hardware/source/checkpoint metadata, quality tables, runtime estimates and figures. Weights and input images are excluded. No GPU execution or speedup is assumed in advance.

In [ ]:
archive_path = Path('/content/da3_gpu_smoke_results.zip')
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for folder in (results_root, project / 'depth_maps/da3_gpu_smoke'):
        if folder.is_dir():
            for file in sorted(folder.rglob('*')):
                if file.is_file():
                    archive.write(file, file.relative_to(project))
print('Download:', archive_path.name, f'({archive_path.stat().st_size / 1024**2:.1f} MiB)')
files.download(str(archive_path))

## Next decision

Return **da3_gpu_smoke_results.zip**. Compare FP16 against FP32 for speed, raw depth quality, shape and coverage on the same frames, then decide which configuration should generate the full sequences. Keep the GPU's hardware label with its timings and compare CPU results separately.

**Validation:** notebook schema, code and embedded inventory/plot scripts are checked locally. Actual Colab allocation, dependency installation, GPU memory fit, inference and browser upload/download remain to be executed in your session.